# Tarea 1

> autor:    Jesus Ivan Rivera Ramirez

> curso:    Inteligencia Artificial

> Trimestre: O26

Considerando el laberinto de la figura 1.

![image.png](resources/laberinto.png)

## Requisitos del programa.
 1. Realice un programa que, utilizando la estrategia de búsqueda primero en anchura, encuentre el
    camino de celdas a seguir para llegar desde una celda origen hasta una celda destino en el  
    laberinto.
    Especificaciones para el programa:
    - Para calcular el costo de los caminos se considerará que cada movimiento tiene un costo de una unidad.
    - Se imprimirá en pantalla la información de los nodos explorados en cada nivel, incluyendo:
        la celda alcanzada, el costo del camino recorrido, el camino recorrido y la profundidad del
        nodo.
    - Al terminar, se imprimirá en pantalla el camino encontrado, especificando claramente las
celdas y el orden en que deben ser visitadas desde la celda origen hasta la celda destino,
así como el costo correspondiente.

# Solución

Comenzaremos modelando el laberinto para ello lo guardaremos en memoria como un arreglo bidimensional, para que los indicies del arreglo 2D correspondan con los estados usaremos las siguientes funciones auxiliares `stateToRowCol` y `rowColToState` que transforman el estado `estado` en el indice `[fila][columna]` y el indice  `[fila][columna]` en el `estado` del laberinto mostrado en la imagen anterior respectivamente.

In [1]:
nRowsMaze = 4 
nColsMaze  = 4
State = int
def stateToRowCol(state: State) -> tuple[int,int]:
    state -= 1
    row = state//nColsMaze
    col = state%nColsMaze
    return (row,col)

def rowColToState(row: int,col:int) -> State:
    return col+row*nColsMaze+1

Podemos probar las funciones anteriores listando las equivalencias entre indices y estados

In [2]:
for i in range(1,17):
    print(f"Estado: {i:>2} = {stateToRowCol(i)}")

Estado:  1 = (0, 0)
Estado:  2 = (0, 1)
Estado:  3 = (0, 2)
Estado:  4 = (0, 3)
Estado:  5 = (1, 0)
Estado:  6 = (1, 1)
Estado:  7 = (1, 2)
Estado:  8 = (1, 3)
Estado:  9 = (2, 0)
Estado: 10 = (2, 1)
Estado: 11 = (2, 2)
Estado: 12 = (2, 3)
Estado: 13 = (3, 0)
Estado: 14 = (3, 1)
Estado: 15 = (3, 2)
Estado: 16 = (3, 3)


De igual forma equivalencias entre indices y estados

In [3]:
for i in range(nRowsMaze):
    for j in range(nColsMaze):
        print(f"Indice: [{i}][{j}] = {rowColToState(i,j)}")

Indice: [0][0] = 1
Indice: [0][1] = 2
Indice: [0][2] = 3
Indice: [0][3] = 4
Indice: [1][0] = 5
Indice: [1][1] = 6
Indice: [1][2] = 7
Indice: [1][3] = 8
Indice: [2][0] = 9
Indice: [2][1] = 10
Indice: [2][2] = 11
Indice: [2][3] = 12
Indice: [3][0] = 13
Indice: [3][1] = 14
Indice: [3][2] = 15
Indice: [3][3] = 16


Para continuar modelando el laberinto definiremos las paredes internas para esto usaremos un diccionario que indica el estado y la posición de la pared respecto a este estado. Las posiciones son:
- **U**: Arriba
- **R**: Derecha
- **D**: Abajo
- **L**: Izquierda


In [4]:
walls = {2:['D'],
         3:['D'],
         5:['R'],
         6:['U','L'],
         7:['U','D'],
         8:['D'],
         9:['R'],
         10:['D','L'],
         11:['U','D'],
         12:['U','D'],
         14:['U'],
         15:['U'],
         16:['U']
         }

Teniendo las parades definidas podemos definir la función sucesor. Usaremos la idea de la cuadricula entonces para obtener los sucesores de un estado es suficiente recuperar los estados vecinos en visualmente en la cuadricula.

In [5]:
def succesor(state: State) -> list[State]:
    stateWalls = walls[state] if state in walls else {}
    row, col = stateToRowCol(state)
    directions = {'U':(0,-1), 'R':(1,0), 'D':(0,1), 'L':(-1,0)}
    succesors = []
    for directionName, direction in directions.items():
        if directionName in stateWalls:
            continue
        dx = direction[0]
        dy = direction[1]
        nRow = row + dy
        nCol = col + dx
        if 0>nRow or nColsMaze<=nRow:
            continue
        if 0>nCol or nRowsMaze<=nCol:
            continue
        succesors.append(rowColToState(nRow,nCol))
    return succesors

Podemos validar que la función `successor` devuelve los estados vecinos esperados.

In [6]:
for i in range(1,17):
    print(f'El estado {i}: tiene los vecinos {succesor(i)}')

El estado 1: tiene los vecinos [2, 5]
El estado 2: tiene los vecinos [3, 1]
El estado 3: tiene los vecinos [4, 2]
El estado 4: tiene los vecinos [8, 3]
El estado 5: tiene los vecinos [1, 9]
El estado 6: tiene los vecinos [7, 10]
El estado 7: tiene los vecinos [8, 6]
El estado 8: tiene los vecinos [4, 7]
El estado 9: tiene los vecinos [5, 13]
El estado 10: tiene los vecinos [6, 11]
El estado 11: tiene los vecinos [12, 10]
El estado 12: tiene los vecinos [11]
El estado 13: tiene los vecinos [9, 14]
El estado 14: tiene los vecinos [15, 13]
El estado 15: tiene los vecinos [16, 14]
El estado 16: tiene los vecinos [15]


Habiendo defino la función sucesor podemos implementar la estructura del nodo, la cual requiere un id, el estado, el costo del camino recorrido, la profunididad del nodo y el camino recorrido tal como lo pide el ejercicio.

In [7]:
from dataclasses import dataclass
@dataclass
class Node:
    state: State 
    path: list[State]

    id: int = 0
    cost: int = 0
    depth: int = 0

    def __str__(self):
        return f"Nodo: {self.id}\n Estado: {self.state}\n Costo: {self.cost}\n Profundidad: {self.depth}"

Ahora podemos generar la función que explora un nodo dado, adicionalmente requiere un conjunto de estados previamente explorados esto para evitar explorar un nodo varias veces. Esto se hace pues nos interesan soluciones optimas y las acciones de ir y regresar a un nodo solo incrementan el costo de una solución por esto no las consideraremos.

In [8]:
def explore(node: Node, seen: set[State])->list[Node]:
    candidates = succesor(node.state)
    res = []
    for i,candidate in enumerate(candidates):
        if candidate in seen:
            continue
        res.append(Node(id    = len(seen), 
                        state = candidate, 
                        cost  = node.cost+1, 
                        depth = node.depth+1,
                        path  = node.path+[candidate]))
        seen.add(candidate)
    return res

Crearemos la función de test la cual nos ayudara a decidir si un nodo de nuestro arbol de busqueda ya se encuentra en el estado objetivo.

In [9]:
def isGoal(node: Node, goal:State):
    return node.state==goal

Juntando todas las piezas podemos crear la busqueda primero en profundidad 

In [12]:

def bfs(start: State, goal: State):
    queue:list[Node] = []
    startNode = Node(state=start, path=[start])
    seen = set()
    print(startNode)
    if isGoal(startNode, goal):
        return startNode
    queue.append(startNode)
    seen.add(start)
    while queue:
        nodeToExplore = queue.pop(0)
        candidates = explore(nodeToExplore, seen)
        for candidate in candidates:
            print(candidate)
            if isGoal(candidate, goal):
                return candidate
            queue.append(candidate) 

In [13]:
startState = 1
endState = 12
res = bfs(startState,endState)

Nodo: 0
 Estado: 1
 Costo: 0
 Profundidad: 0
Nodo: 1
 Estado: 2
 Costo: 1
 Profundidad: 1
Nodo: 2
 Estado: 5
 Costo: 1
 Profundidad: 1
Nodo: 3
 Estado: 3
 Costo: 2
 Profundidad: 2
Nodo: 4
 Estado: 9
 Costo: 2
 Profundidad: 2
Nodo: 5
 Estado: 4
 Costo: 3
 Profundidad: 3
Nodo: 6
 Estado: 13
 Costo: 3
 Profundidad: 3
Nodo: 7
 Estado: 8
 Costo: 4
 Profundidad: 4
Nodo: 8
 Estado: 14
 Costo: 4
 Profundidad: 4
Nodo: 9
 Estado: 7
 Costo: 5
 Profundidad: 5
Nodo: 10
 Estado: 15
 Costo: 5
 Profundidad: 5
Nodo: 11
 Estado: 6
 Costo: 6
 Profundidad: 6
Nodo: 12
 Estado: 16
 Costo: 6
 Profundidad: 6
Nodo: 13
 Estado: 10
 Costo: 7
 Profundidad: 7
Nodo: 14
 Estado: 11
 Costo: 8
 Profundidad: 8
Nodo: 15
 Estado: 12
 Costo: 9
 Profundidad: 9


In [ ]:
print(f'La ruta desde el estado {startState} hasta el estado {endState} es: {"->".join(map(str, res.path))}.')

La ruta desde el estado 1 hasta el estado 12 es: 1->2->3->4->8->7->6->10->11->12.
